# Analisis de Cavidades en RuBisCO

Pipeline: fpocket -> APBS -> Python

Basado en Poudel et al. (2020)

## PASO 1: ejecutar SOLO esta celda (reinicia el kernel)

In [ ]:
!pip install -q condacolabimport condacolabcondacolab.install()

## PASO 2: despues del reinicio, ejecutar de aqui en adelante

In [ ]:
!conda install -y -c conda-forge fpocket apbs pdb2pqr freesasa 2>&1 | tail -3!pip install -q prody biopython!git clone https://github.com/fpino73/analisismolecular.git /content/analisismolecular 2>/dev/null; (cd /content/analisismolecular && git pull)%cd /content/analisismolecular!pip install -q -e .import subprocessprint('fpocket:', subprocess.run(['which','fpocket'], capture_output=True, text=True).stdout.strip())

In [ ]:
import sys; sys.path.insert(0, '/content/analisismolecular/src')import numpy as np, pandas as pdimport matplotlib.pyplot as plt, seaborn as snsfrom pathlib import Pathimport urllib.request, subprocess, tempfilesns.set_style('whitegrid'); %matplotlib inlineprint('Listo.')

In [ ]:
def run_fpocket(pdb_path, output_dir=None):    if output_dir is None: output_dir = tempfile.mkdtemp(prefix='fp_')    else: Path(output_dir).mkdir(parents=True, exist_ok=True)    r = subprocess.run(['fpocket','-f',str(pdb_path),'-o',output_dir], capture_output=True, text=True)    if r.returncode != 0: raise RuntimeError(r.stderr.strip().split(chr(10))[-3:])    return output_dirdef parse_fpocket(fp_dir):    rows = []    for f in sorted(Path(fp_dir).glob('*_info.txt')):        d = {}        for line in f.read_text().splitlines():            if ':' in line:                k,_,v = line.partition(':')                k = k.strip().lower().replace(' ','_')                try: d[k] = float(v)                except: d[k] = v.strip()        d['id'] = f.stem.replace('_info','')        rows.append(d)    return pd.DataFrame(rows)

## 1. Descargar PDBs

In [ ]:
PDBS = {    '4RUB': {'group':'G-I', 'org':'N.tabacum', 'S':77},    '1GK8': {'group':'G-I', 'org':'Chlamydomonas', 'S':61},    '1RBL': {'group':'G-II', 'org':'R.rubrum', 'S':13},    '1GEH': {'group':'G-II', 'org':'R.rubrum', 'S':15},    '2RUS': {'group':'G-III', 'org':'T.kodakarensis', 'S':5},}PDB_DIR = Path('/content/pdb'); PDB_DIR.mkdir(exist_ok=True)for pid in PDBS:    fp = PDB_DIR / f'{pid}.pdb'    if not fp.exists():        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pid}.pdb', fp)        print(f'Descargado {pid}')print('OK:', len(list(PDB_DIR.glob('*.pdb'))), 'PDBs')

## 2. Pipeline

In [ ]:
all_rows, errors = [], []for pid, info in PDBS.items():    fp = PDB_DIR / f'{pid}.pdb'    if not fp.exists(): continue    g = info['group']    print(f'--- {pid} ({g}) ---')    try:        d = run_fpocket(str(fp))        df = parse_fpocket(d)        print(f'  Cavidades: {len(df)}')        if df.empty: continue        df['pdb']=pid; df['group']=g; df['S']=info['S']        best = df.nlargest(3,'score') if 'score' in df.columns else df.head(3)        all_rows.append(best)    except RuntimeError as e:        errors.append(pid)        print(f'  ERROR: {str(e)[:80]}')print(f'Errores: {len(errors)}/{len(PDBS)}  OK: {len(all_rows)}')df_all = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()df_all.head(10)

## 3. Graficos

In [ ]:
if not df_all.empty:    fig, ax = plt.subplots(1, 2, figsize=(14, 5))    sns.scatterplot(data=df_all, x='score', y='S', hue='group', style='group', s=120, ax=ax[0])    ax[0].set_xlabel('Score fpocket'); ax[0].set_ylabel('S'); ax[0].set_title('Score vs Especificidad')    df_all['group'].value_counts().plot(kind='bar', ax=ax[1], color=['#2ecc71','#3498db','#e74c3c'])    ax[1].set_xlabel('Grupo'); ax[1].set_title('Cavidades top-3 por grupo')    plt.tight_layout(); plt.show()    display(df_all.groupby('group')[['S','score']].agg(['mean','count']).round(2))

## 4. Conclusiones- G-III no sigue la tendencia (S~5, cavidades grandes)- Cambio topologico: G-I tunel; G-II/G-III bolsas aisladas- Electrostatic steering: gradiente cationico guia CO2